# H1 · Estadística descriptiva con formularios

**Estadística I · Ingeniería Industrial · Universidad Industrial de Santander**  
*Herramienta*

**Propósito de aprendizaje**

- Explorar cualquier base de datos (CSV o Excel) sin escribir código, llenando formularios.
- Construir tablas de frecuencia, medidas resumen y gráficos univariados y bivariados.
- Servir de apoyo en clase o en proyectos: el estudiante se concentra en interpretar.

**Duración sugerida:** Uso libre  
**Datos:** cualquier base del repositorio (por defecto `seguro_medico.csv`)

## Cómo usar este notebook

1. Abra el notebook en **Google Colab**. Las celdas tienen formularios a la derecha: cambie los valores y ejecute la celda (Shift + Enter).
2. Ejecute primero la sección **Lectura de datos**. Las demás celdas usan la tabla `datos` que allí se crea.
3. Los valores por defecto usan `seguro_medico.csv` como ejemplo. Cambie el nombre del archivo y de las variables para trabajar con otra base.

> En Jupyter (fuera de Colab) los formularios no se ven, pero las celdas funcionan igual: cambie el valor de la variable en el código.

# 1. Lectura y preparación de datos

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_rows", 150)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", palette=["#00ADC6", "#FF6F49", "#25A687", "#7B6FD0", "#F2B134"])

def leer_datos(nombre, hoja=None):
    """Lee CSV o Excel. Busca en la carpeta actual y en datos/ del repositorio; en Colab ofrece subir el archivo."""
    rutas = [Path(nombre)] + [Path(p) / "datos" / c / nombre for p in [".", "..", "../.."] for c in ["clasicos", "casos_industriales/csv", "casos_industriales/xlsx", "talleres"]]
    ruta = next((r for r in rutas if r.exists()), None)
    if ruta is None:
        try:
            from google.colab import files
            print(f"No encontré '{nombre}'. Súbalo:")
            ruta = Path(next(iter(files.upload())))
        except ImportError:
            raise FileNotFoundError(f"No encontré '{nombre}'.")
    if ruta.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(ruta, sheet_name=hoja or 0)
    return pd.read_csv(ruta, sep=None, engine="python")  # detecta coma o punto y coma

In [ ]:
#@markdown ### **Cargar el archivo**
archivo = "seguro_medico.csv"  #@param {type:"string"}
hoja_excel = ""  #@param {type:"string"}
#@markdown ---
datos_originales = leer_datos(archivo, hoja_excel or None)
datos = datos_originales.copy()
print(f"Se cargaron {len(datos)} registros y {datos.shape[1]} variables.")
datos.head()

In [ ]:
#@markdown ### **Ver tipos de variables y datos faltantes**
resumen = pd.DataFrame({"tipo": datos.dtypes.astype(str),
                        "no_nulos": datos.notna().sum(),
                        "faltantes": datos.isna().sum(),
                        "valores_distintos": datos.nunique()})
resumen

In [ ]:
#@markdown ### **Renombrar columnas (opcional)**
#@markdown Escriba los nuevos nombres separados por comas, en el mismo orden. Déjelo vacío para no cambiar nada.
nuevos_nombres = ""  #@param {type:"string"}
#@markdown ---
if nuevos_nombres.strip():
    nombres = [c.strip() for c in nuevos_nombres.split(",")]
    if len(nombres) != datos.shape[1]:
        print(f"Escribió {len(nombres)} nombres, pero la base tiene {datos.shape[1]} columnas. No se cambió nada.")
    else:
        datos.columns = nombres
print(list(datos.columns))

In [ ]:
#@markdown ### **Filtrar los datos (opcional)**
#@markdown Ejemplos: `edad >= 30`, `fumador == "si"`, `region in ["noreste", "noroeste"]`. Vacío = sin filtro.
condicion = ""  #@param {type:"string"}
#@markdown ---
# El filtro se aplica sobre la tabla ya preparada (no vuelve a leer el archivo).
datos = datos.query(condicion) if condicion.strip() else datos
print(f"Registros después del filtro: {len(datos)}")

---
# 2. Análisis univariado
## 2.1 Variable categórica

In [ ]:
#@markdown ### **Tabla de frecuencias**
variable = "region"  #@param {type:"string"}
absoluta = True  #@param {type:"boolean"}
relativa = True  #@param {type:"boolean"}
porcentual = True  #@param {type:"boolean"}
ordenar_por = "frecuencia"  #@param ["frecuencia", "categoria"]
#@markdown ---
conteo = datos[variable].value_counts(dropna=False)
if ordenar_por == "categoria":
    conteo = conteo.sort_index()
tabla = pd.DataFrame(index=conteo.index)
if absoluta:   tabla["absoluta"] = conteo
if relativa:   tabla["relativa"] = (conteo / conteo.sum()).round(4)
if porcentual: tabla["porcentual"] = (100 * conteo / conteo.sum()).round(2)
tabla.loc["Total"] = tabla.sum()
if absoluta: tabla["absoluta"] = tabla["absoluta"].astype(int)
tabla

In [ ]:
#@markdown ### **Diagrama de barras**
variable = "region"  #@param {type:"string"}
frecuencia = "porcentual"  #@param ["absoluta", "porcentual"]
orientacion = "horizontal"  #@param ["horizontal", "vertical"]
ordenar_por = "frecuencia"  #@param ["frecuencia", "categoria"]
titulo = ""  #@param {type:"string"}
#@markdown ---
orden = datos[variable].value_counts().index if ordenar_por == "frecuencia" else sorted(datos[variable].dropna().unique())
stat = "count" if frecuencia == "absoluta" else "percent"
eje = {"y": variable} if orientacion == "horizontal" else {"x": variable}
ax = sns.countplot(data=datos, order=orden, stat=stat, color="#00ADC6", **eje)
ax.set_title(titulo or f"Distribución de {variable}")
ax.set_xlabel("Porcentaje" if (stat == "percent" and orientacion == "horizontal") else ax.get_xlabel())
plt.show()

## 2.2 Variable numérica

In [ ]:
#@markdown ### **Distribución de frecuencias por intervalos**
variable = "cargos_seguro_medico"  #@param {type:"string"}
#@markdown Número de intervalos (0 = regla de Sturges) y límite inferior (vacío = mínimo de los datos)
k = 0  #@param {type:"integer"}
limite_inferior = ""  #@param {type:"string"}
amplitud = ""  #@param {type:"string"}
#@markdown ---
x = datos[variable].dropna()
k_uso = k if k > 0 else int(np.ceil(1 + np.log2(len(x))))
li = float(limite_inferior) if limite_inferior.strip() else x.min()
A = float(amplitud) if amplitud.strip() else (x.max() - li) / k_uso
limites = li + A * np.arange(k_uso + 1)
limites[-1] = max(limites[-1], x.max()) + 1e-9          # incluye el máximo
intervalos = pd.cut(x, limites, right=False)
f = intervalos.value_counts().sort_index()
tabla = pd.DataFrame({"marca_clase": [iv.mid for iv in f.index],
                      "frec_absoluta": f.values,
                      "frec_relativa": (f / f.sum()).round(4).values,
                      "frec_abs_acumulada": f.cumsum().values,
                      "frec_rel_acumulada": (f.cumsum() / f.sum()).round(4).values}, index=f.index.astype(str))
print(f"k = {k_uso} intervalos de amplitud {A:,.2f}")
tabla

In [ ]:
#@markdown ### **Medidas resumen**
variable = "cargos_seguro_medico"  #@param {type:"string"}
#@markdown ---
x = datos[variable].dropna()
q1, q2, q3 = x.quantile([.25, .5, .75])
medidas = pd.Series({
    "n": x.count(), "mínimo": x.min(), "Q1": q1, "mediana": q2, "Q3": q3, "máximo": x.max(),
    "rango": x.max() - x.min(), "RIC": q3 - q1,
    "media": x.mean(), "desviación estándar": x.std(), "varianza": x.var(),
    "coef. de variación (%)": 100 * x.std() / x.mean(),
    "asimetría": x.skew(),
    "curtosis (exceso)": x.kurt(),   # 0 para la normal; > 0 colas más pesadas
}, name=variable)
medidas.to_frame().style.format("{:,.3f}")

In [ ]:
#@markdown ### **Histograma**
variable = "cargos_seguro_medico"  #@param {type:"string"}
intervalos = 0  #@param {type:"integer"}
frecuencia = "percent"  #@param ["count", "percent", "density"]
mostrar_curva_densidad = True  #@param {type:"boolean"}
escala_log_x = False  #@param {type:"boolean"}
#@markdown ---
x = datos[variable].dropna()
bins = intervalos if intervalos > 0 else "sturges"
ax = sns.histplot(x, bins=bins, stat=frecuencia, kde=mostrar_curva_densidad, color="#00ADC6", log_scale=escala_log_x)
ax.axvline(x.mean(), color="#FF6F49", ls="--", label=f"media = {x.mean():,.1f}")
ax.axvline(x.median(), color="#25A687", ls=":", label=f"mediana = {x.median():,.1f}")
ax.legend(); ax.set_title(f"Histograma de {variable}")
plt.show()

In [ ]:
#@markdown ### **Diagrama de caja y bigotes**
variable = "cargos_seguro_medico"  #@param {type:"string"}
orientacion = "horizontal"  #@param ["horizontal", "vertical"]
#@markdown ---
x = datos[variable].dropna()
ax = sns.boxplot(x=x, color="#00ADC6", width=.35) if orientacion == "horizontal" else sns.boxplot(y=x, color="#00ADC6", width=.35)
ax.set_title(f"Caja y bigotes de {variable}")
q1, q3 = x.quantile([.25, .75]); ric = q3 - q1
atipicos = x[(x < q1 - 1.5 * ric) | (x > q3 + 1.5 * ric)]
print(f"Límites de Tukey: [{q1 - 1.5*ric:,.2f} ; {q3 + 1.5*ric:,.2f}]  ->  {len(atipicos)} datos atípicos ({100*len(atipicos)/len(x):.1f} %)")
plt.show()

---
# 3. Análisis bivariado
## 3.1 Dos variables categóricas

In [ ]:
#@markdown ### **Tabla de contingencia**
fila = "region"  #@param {type:"string"}
columna = "fumador"  #@param {type:"string"}
frecuencia = "porcentaje por fila"  #@param ["absoluta", "porcentaje del total", "porcentaje por fila", "porcentaje por columna"]
#@markdown ---
norm = {"absoluta": False, "porcentaje del total": "all", "porcentaje por fila": "index", "porcentaje por columna": "columns"}[frecuencia]
tabla = pd.crosstab(datos[fila], datos[columna], margins=True, margins_name="Total", normalize=norm)
if norm is not False:
    tabla = (100 * tabla).round(2)
tabla

In [ ]:
#@markdown ### **Barras agrupadas**
fila = "region"  #@param {type:"string"}
columna = "fumador"  #@param {type:"string"}
#@markdown ---
p = pd.crosstab(datos[fila], datos[columna], normalize="index") * 100
ax = p.plot(kind="bar", stacked=True, color=["#00ADC6", "#FF6F49", "#25A687", "#7B6FD0", "#F2B134"][:p.shape[1]], figsize=(8, 4.5))
ax.set_ylabel(f"% dentro de cada {fila}"); ax.set_title(f"{columna} según {fila}")
ax.legend(title=columna, bbox_to_anchor=(1.02, 1), loc="upper left"); plt.xticks(rotation=0)
plt.show()

## 3.2 Variable numérica según una categórica

In [ ]:
#@markdown ### **Medidas resumen por grupo**
variable = "cargos_seguro_medico"  #@param {type:"string"}
grupos = "fumador"  #@param {type:"string"}
#@markdown ---
g = datos.groupby(grupos)[variable]
pd.DataFrame({"n": g.count(), "media": g.mean(), "mediana": g.median(), "desv_est": g.std(),
              "CV_%": 100 * g.std() / g.mean(), "Q1": g.quantile(.25), "Q3": g.quantile(.75),
              "asimetria": g.skew()}).round(2)

In [ ]:
#@markdown ### **Cajas por grupo**
variable = "cargos_seguro_medico"  #@param {type:"string"}
grupos = "fumador"  #@param {type:"string"}
escala_log = False  #@param {type:"boolean"}
#@markdown ---
ax = sns.boxplot(data=datos, x=grupos, y=variable, color="#00ADC6", log_scale=escala_log)
sns.stripplot(data=datos, x=grupos, y=variable, color="#333", alpha=.15, size=2, ax=ax)
ax.set_title(f"{variable} según {grupos}")
plt.show()

## 3.3 Dos variables numéricas

In [ ]:
#@markdown ### **Diagrama de dispersión**
variable_x = "edad"  #@param {type:"string"}
variable_y = "cargos_seguro_medico"  #@param {type:"string"}
color_por = "fumador"  #@param {type:"string"}
recta_de_tendencia = True  #@param {type:"boolean"}
#@markdown ---
ax = sns.scatterplot(data=datos, x=variable_x, y=variable_y, hue=color_por or None, alpha=.6)
if recta_de_tendencia:
    sns.regplot(data=datos, x=variable_x, y=variable_y, scatter=False, color="#333", ax=ax, line_kws={"lw": 1.5, "ls": "--"})
ax.set_title(f"{variable_y} vs. {variable_x}")
plt.show()

In [ ]:
#@markdown ### **Coeficientes de correlación**
variable_1 = "edad"  #@param {type:"string"}
variable_2 = "cargos_seguro_medico"  #@param {type:"string"}
#@markdown ---
par = datos[[variable_1, variable_2]].dropna()
r_p = par[variable_1].corr(par[variable_2], method="pearson")
r_s = par[variable_1].corr(par[variable_2], method="spearman")
print(f"Pearson  r   = {r_p:.4f}   (relación lineal)")
print(f"Spearman rho = {r_s:.4f}   (relación monótona, robusta a atípicos)")
print(f"n = {len(par)} pares completos")

In [ ]:
#@markdown ### **Matriz de correlación de todas las variables numéricas**
metodo = "pearson"  #@param ["pearson", "spearman"]
#@markdown ---
m = datos.select_dtypes("number").corr(method=metodo)
plt.figure(figsize=(1 + .8 * len(m), .8 * len(m)))
sns.heatmap(m, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
plt.title(f"Correlación de {metodo.capitalize()}")
plt.show()